In [1]:
import os
import pandas as pd
import numpy as np
from tokenizers import Tokenizer, models

In [2]:
DATA_DIR = "/home/mkeoliya/projects/CAREBench/data"

In [4]:
FNAME = 'ehrshot'

In [5]:
if FNAME == 'ehrshot':
    numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot', 'numerics.csv'), usecols=['eventval'])
    med_df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot', 'med.csv'), usecols=['eventval'])  
    df = pd.concat([med_df, numerics_df], ignore_index=True)  
    demo_df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot', 'demo.csv'), usecols=['Gender_str', 'Race_str', 'Ethnicity_str', 'Age_str'])
elif FNAME == 'mcmed':
    labs_df = pd.read_csv(os.path.join(DATA_DIR, 'mcmed', 'labs.csv'), usecols=['eventval'])   
    numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'mcmed', 'numerics.csv'), usecols=['eventval'])
    orders_df = pd.read_csv(os.path.join(DATA_DIR, 'mcmed', 'orders.csv'), usecols=['eventval'])  
    df = pd.concat([labs_df, numerics_df, orders_df], ignore_index=True) 
    demo_df = pd.read_csv(os.path.join(DATA_DIR, 'mcmed', 'demo.csv'), usecols=['Gender_str', 'Race_str', 'Ethnicity_str', 'Age_str'])  
elif FNAME == 'mimiced':
    demo_df = pd.read_csv(os.path.join(DATA_DIR, 'mimiciv/3.1/ed', 'demo.csv'), usecols=['Gender_str', 'Race_str'])  
    numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'mimiciv/3.1/ed', 'numerics.csv'), usecols=['eventval'])
    med_df = pd.read_csv(os.path.join(DATA_DIR, 'mimiciv/3.1/ed', 'med.csv'), usecols=['eventval'])  
    df = pd.concat([med_df, numerics_df], ignore_index=True)  

CREATE TOKENIZER

In [6]:
vocab_list = list(df['eventval'].astype(str).unique())
vocab_list += list(demo_df['Gender_str'].unique()) + list(demo_df['Race_str'].unique()) + list(demo_df['Ethnicity_str'].unique()) + list(demo_df['Age_str'].unique())

In [7]:
vocab_list = [x for x in vocab_list if x is not None]
vocab_list = list(set(vocab_list))

special_list = ['[UNK]', '[BOS]', '[EOS]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
vocab_list = special_list + vocab_list

# Create a dictionary mapping each token to a unique index
vocab = {token: idx for idx, token in enumerate(vocab_list)}

# Create a WordLevel tokenizer with the custom vocab and an unknown token
tokenizer = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]"))

tokenizer.save(os.path.join(DATA_DIR, f'{FNAME}_tokenizer.json'))